# 12. Quantification reference analysis

This notebook establishes the first defensible quantification layer for the restored SD302 cohort. It addresses the three requested parameters separately: **pattern intensity**, **minutiae**, and **total finger ridge count (TFRC)**.

The analysis uses examiner annotations where they exist, selects one uniform ten-finger acquisition series, and does not present an unvalidated image-processing output as an accurate ridge count.

## Analysis rules

1. Canonical impressions are the SD302b baseline device V, 1000-ppi rolled images.
2. Pattern intensity requires ten available and classifiable finger patterns.
3. Minutiae totals use examiner-marked EFS field 9.331 features.
4. Subject-linked rows remain private; only aggregate tables and figures are exported.
5. TFRC is treated as outside the primary reported endpoint because validated expert field 9.322 ridge-count annotations are absent from the restored records.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RESULTS = ROOT / 'results' / 'quantification_reference'
PRIVATE = ROOT / 'data' / 'processed' / 'sd302_2026_restoration' / 'quantification_reference'
RESULTS.mkdir(parents=True, exist_ok=True)

## 1. Rebuild and verify the reference tables

The builder applies all cohort rules before this notebook performs interpretation. Re-running it makes the notebook independent of stale aggregate files.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, str(ROOT / 'scripts' / 'build_quantification_reference.py')],
    cwd=ROOT, check=True,
)
manifest = json.loads((RESULTS / 'manifest.json').read_text(encoding='utf-8'))
manifest

In [ ]:
coverage = pd.read_csv(RESULTS / 'coverage_summary.csv')
coverage

The canonical series contains 1,999 impressions from 200 subjects. One subject lacks a tenth image. Eleven complete-image subjects contain an examiner `UC` code, so they remain available for minutiae totals but are excluded from pattern intensity.

## 2. Pattern intensity

For a ten-finger set, the individual pattern intensity index is `number of loops + 2 x number of whorls`, producing a range from 0 to 20. This is equivalent to the percentage-based formula reported in dermatoglyphic studies: `(2 x % whorls + % loops) / 10`. Arches contribute zero. See the [published formula and pattern definitions](https://pmc.ncbi.nlm.nih.gov/articles/PMC6967092/).

In [ ]:
pii_summary = pd.read_csv(RESULTS / 'pattern_intensity_summary.csv')
pii_distribution = pd.read_csv(RESULTS / 'pattern_intensity_distribution.csv')
display(pii_summary)
display(pii_distribution.T)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(pii_distribution['pattern_intensity_index'], pii_distribution['subjects'], color='#176b5b')
ax.set(xlabel='Pattern intensity index (0-20)', ylabel='Subjects', title='Ten-finger pattern intensity distribution')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(RESULTS / 'pattern_intensity_distribution.png', dpi=180)
plt.show()

## 3. Examiner-marked minutiae

Field 9.331 distinguishes ridge endings and bifurcations. These counts are valid summaries of the supplied EFS annotations, not estimates of every biological minutia present on a finger. They can vary with image coverage, quality, device, and examiner markup protocol.

In [ ]:
ten_finger_minutiae = pd.read_csv(RESULTS / 'ten_finger_minutiae_summary.csv')
minutiae_by_pattern = pd.read_csv(RESULTS / 'minutiae_by_pattern.csv')
display(ten_finger_minutiae)
display(minutiae_by_pattern)

In [ ]:
canonical = pd.read_csv(PRIVATE / 'canonical_finger_quantification.csv')
order = ['arch', 'left_slant_loop', 'right_slant_loop', 'whorl']
groups = [canonical.loc[canonical['broad_class'] == label, 'minutiae_count'] for label in order]
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.boxplot(groups, tick_labels=[label.replace('_', ' ') for label in order], showfliers=False)
ax.set(ylabel='Examiner-marked minutiae per image', title='Minutiae annotation counts by broad pattern')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(RESULTS / 'minutiae_by_pattern.png', dpi=180)
plt.show()

## 4. Repeat-impression sensitivity

Where another device captured the same subject and finger, its annotations are compared with the canonical device V record. This is a sensitivity analysis, not a formal inter-rater reliability study: devices and image conditions differ.

In [ ]:
repeat_agreement = pd.read_csv(RESULTS / 'repeat_impression_agreement.csv')
repeat_agreement

Broad-pattern agreement is high across paired impressions, but minutiae totals are not identical. The overall median absolute difference should be reported whenever annotation totals are interpreted.

## 5. Total finger ridge count status

ANSI/NIST field 9.322 stores core-to-delta ridge counts, but a complete scan found that field absent from all 2,380 SD302g IRR records. The standard defines a count as the intervening ridges crossed by a straight core-to-delta line, excluding the core and delta ridges ([NIST guidance](https://tsapps.nist.gov/trainingtool/AdditionalFeatures/CoreDeltaRidge.html)). Conventional TFRC sums one count per finger and normally uses the larger count for a whorl.

The restored cores, deltas, and images make an image-derived estimate feasible as a separate methodological study. In this phase, TFRC is not reported as a primary quantitative outcome because the restored expert annotations do not include validated ridge-count values.

In [ ]:
decisions = pd.DataFrame([
    {'parameter': 'Pattern intensity', 'current status': 'Calculated', 'eligible subjects': 188, 'interpretation': 'Ten-finger qualitative pattern index'},
    {'parameter': 'Minutiae', 'current status': 'Calculated', 'eligible subjects': 199, 'interpretation': 'Examiner-marked EFS feature totals'},
    {'parameter': 'Total finger ridge count', 'current status': 'Not primary endpoint', 'eligible subjects': 0, 'interpretation': 'Validated expert ridge-count field 9.322 absent from restored annotations'},
])
decisions.to_csv(RESULTS / 'quantification_status.csv', index=False)
decisions

## Research conclusion

The project now has an auditable pattern-intensity measure and examiner-annotation minutiae summaries. These are descriptive research quantities and are not identity determinations or diagnostic measures. TFRC remains a candidate endpoint outside the primary reported quantification layer for this phase because validated expert ridge-count annotations were not present in the restored SD302g records.